In [ ]:
# %%
import os
import pandas as pd
import json
import subprocess
from datetime import datetime
import re
import folium

# Paths

In [ ]:
JSON_PATH = r"\\192.168.20.101\webdata\Nav\old"

In [ ]:
"""Write a program to get all .json files in JSON_PATH and drop duplicates using serverTime and get results to a panda dataframe"""
# %%


In [ ]:
# Get all .json files in JSON_PATH and drop duplicates using serverTime
json_files = [
    os.path.join(JSON_PATH, file)
    for file in os.listdir(JSON_PATH)
    if file.endswith(".json")
]

# Combine all JSON files into a single dataframe
dataframes = []
for file in json_files:
    with open(file, "r") as f:
        data = json.load(f)
        dataframes.append(pd.DataFrame(data))

# Concatenate all dataframes
combined_df = pd.concat(dataframes, ignore_index=True)

# Drop duplicates based on 'serverTime' column
if "serverTime" in combined_df.columns:
    combined_df = combined_df.drop_duplicates(subset="serverTime")

print(combined_df)

In [ ]:
os.path.join("./output", "combined_data.parquet")

In [ ]:
combined_df.dtypes

In [ ]:
# Ensure latitude and longitude columns are numeric
combined_df["latitude"] = pd.to_numeric(combined_df["latitude"], errors="coerce")
combined_df["longitude"] = pd.to_numeric(combined_df["longitude"], errors="coerce")

# Ensure all columns have consistent data types
combined_df["address"] = combined_df["address"].astype(str)
combined_df["old_latitude"] = combined_df["old_latitude"].astype(str)
combined_df["old_longitude"] = combined_df["old_longitude"].astype(str)

# Drop rows with invalid latitude or longitude
combined_df = combined_df.fillna(0)

# Ensure the 'lat,long' column is properly formatted as a string
combined_df["lat,long"] = combined_df["lat,long"].astype(str)

# Save the dataframe to a Parquet file
combined_df.to_parquet(os.path.join("./output", "combined_data.parquet"), index=False)

In [ ]:
combined_df.head(2)

In [ ]:
combined_df.sort_values(by="serverTime", ascending=False, inplace=True)
combined_df.to_parquet(os.path.join("./output", "combined_data.parquet"), index=False)

In [ ]:
DIAMETER_KMS = 1

In [ ]:
CENTER_LAT, CENTER_LON = 6.15878, 80.0934617

In [ ]:
''' DIAMETER_KMS is giben in kilometers. filter combined_df dataframe so that locations 
identified by  latitude and longitude columns are outside the circle with center
(CENTER_LAT, CENTER_LON) and radius DIAMETER_KMS. '''

In [ ]:
import numpy as np


# Function to calculate the Haversine distance
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in kilometers
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c


# Calculate distances from the center
combined_df["distance_from_center"] = haversine(
    combined_df["latitude"], combined_df["longitude"], CENTER_LAT, CENTER_LON
)

# Filter rows where the distance is greater than the radius
filtered_df = combined_df[combined_df["distance_from_center"] > DIAMETER_KMS / 2]
filtered_df = filtered_df[filtered_df["latitude"] > 0]
filtered_df = filtered_df[filtered_df["longitude"] > 0]
print(filtered_df)

In [ ]:
filtered_df.head(2)

In [ ]:
"""show filtered_df on a map using folium. The icons should be numbered from 1 to n, where n is the number of rows in filtered_df.
When clicked on an icon it should show all the information in the row in a popup.
It should also have clickable link to open the location on Google Maps."""

In [ ]:
map_ = folium.Map(
    location=[filtered_df["latitude"].mean(), filtered_df["longitude"].mean()],
    zoom_start=15,
)

In [ ]:
for i, row in filtered_df.iterrows():
    popup_content = f"""
    <b>Index:</b> {i}<br>
    <b>Latitude:</b> {row['latitude']}<br>
    <b>Longitude:</b> {row['longitude']}<br>
    <b>Address:</b> {row['address']}<br>
    <b>Local Time:</b> {row['localTime']}<br>
    <b>Battery Level:</b> {row['batteryLevel']}<br>
    <b>Google Account:</b> {row['googleAccount']}<br>
    <b>Last Update:</b> {row['lastUpdate']}<br>
    <b>Server Time:</b> {row['serverTime']}<br>
    <b>Distance from Center:</b> {row['distance_from_center']} km<br>
    <a href="https://www.google.com/maps?q={row['latitude']},{row['longitude']}" target="_blank">Open in Google Maps</a>
    """
    # folium.Marker(
    #     location=[row["latitude"], row["longitude"]],
    #     popup=folium.Popup(popup_content, max_width=300),
    #     icon=folium.Icon(color="blue", icon="info-sign"),
    #     tooltip=f"Point {i + 1}",
    # ).add_to(map_)
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=2,  # Adjust the size of the dot
        color="blue" if idx > 0 else "red",  # Outline color
        fill=True,
        fill_color="blue" if idx > 0 else "red",  # Fill color
        fill_opacity=0.1,  # Adjust opacity
        tooltip=f"Point {i + 1}",
        opup=folium.Popup(popup_content, max_width=300),
    ).add_to(map_)

In [ ]:
map_.save(os.path.join("./output", "location_list.html"))

In [ ]:
webbrowser.open(os.path.join("./output", "location_list.html"))